# Yapay Sinir Ağları — Aşama 1 / Sıfırdan Başlangıç (Seed 31040)

**Yeni çalışma / 18 Kasım 2026.** Ödevde istenen yalnızca `bank-additional.csv` kullanılır. Eski deney/test çıktıları tekrar kullanılmaz.

**Yürütme:** Google Colab → Çalışma Zamanı → Tümünü Çalıştır. Google Drive bağlantısı ilk kullanımda sizin onayınızı gerektirir. Çıktılar `MyDrive/Yapay Sinir Ağları/YSA_31040_20261118/` altında kalıcı saklanır.

**Kural:** Aşama 1 ve 2'de test etiketleriyle sonuç metrikleri üretmeyin. Bu notebook test etiketlerini yalnızca stratified bölünme ve sınıf oranı kaydı için kullanır; EDA yalnızca development üzerinde yapılır.


### 0. Google Drive çalışma alanını bağlama

Colab oturumunda Google Drive erişimine **siz** onay verirsiniz. Bu notebook eski proje klasörünüzü değiştirmeden **yeni** `YSA_31040_20261118` klasöründe çalışır. Birden çok kez çalıştırıldığında daha önce kaydedilmiş veri bölünmesinin farklılaşmasını önlemek için kontrol uygular.

In [ ]:
import io, os, sys, json, time, random, zipfile, urllib.request, shutil, warnings, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
import sklearn

# Colab'da Drive'ı kullanıcı oturumu ile bağla. Yerel Jupyter'de çalışma dizinine yaz.
if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    if not DRIVE_ROOT.exists():
        raise RuntimeError('Google Drive bağlanamadı. Erişimi Google hesabınızdan onaylayın.')
    ROOT = DRIVE_ROOT / 'Yapay Sinir Ağları' / 'YSA_31040_20261118'
else:
    ROOT = Path.cwd() / 'YSA_31040_20261118'
DATA_DIR = ROOT / 'data'
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
CONFIGS = ROOT / 'configs'
for folder in [DATA_DIR, RESULTS, FIGURES, CONFIGS]:
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
print('YENİ PROJE KLASÖRÜ:', ROOT)
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)


### 1. Sabit ve tekrarlanabilir seed

Öğrenci numaranızın son beş hanesi **29014** olduğundan `BASE_SEED = 29014 + 2026 = 31040`. Seed değerleri projede kaydedilir. Çalışan notebook bu değerleri değiştirmez.

In [ ]:
STUDENT_LAST5 = '29014'
BASE_SEED = 31040
assert int(STUDENT_LAST5) + 2026 == BASE_SEED
SEEDS = {
    'base': BASE_SEED,
    'python': BASE_SEED + 11,
    'numpy': BASE_SEED + 23,
    'torch': BASE_SEED + 37,
    'split': BASE_SEED + 53,
    'outer_cv': BASE_SEED + 71,
    'inner_cv': BASE_SEED + 89,
}
random.seed(SEEDS['python'])
np.random.seed(SEEDS['numpy'])
try:
    import torch
    torch.manual_seed(SEEDS['torch'])
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEEDS['torch'])
    print('PyTorch:', torch.__version__)
except ImportError:
    print('PyTorch kurulu değil; EDA için zorunlu değil.')
seed_file = CONFIGS / 'seeds.json'
if seed_file.exists():
    previous = json.loads(seed_file.read_text(encoding='utf-8'))
    if previous != SEEDS:
        raise RuntimeError('Kayıtlı seed değerleri değişmiş! Çalışmayı durdurun.')
else:
    seed_file.write_text(json.dumps(SEEDS, indent=2), encoding='utf-8')
print('Temel seed:', BASE_SEED, '| split seed:', SEEDS['split'])


### 2. UCI'dan yalnızca doğru CSV'yi getirme
Kod önce UCI'nın resmî arşivini dener. Ağ erişimi engellenirse Colab'a `bank-additional.csv`, `bank-additional.zip` veya UCI'nın tüm ZIP arşivini yüklemeniz istenir. İç içe ZIP olsa bile **yalnızca adı tam `bank-additional.csv` olan dosya** kullanılır.
**Not:** Resmî arşiv erişimi başarısızsa yalnızca doğru `bank-additional.csv` veya bu dosyayı içeren ZIP yükleyin. `bank-additional-full.csv` kabul edilmez.

In [ ]:
OFFICIAL_URLS = [
    'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip',
    'https://archive.ics.uci.edu/static/public/222/bank%2Bmarketing.zip',
    'https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip',
]

def extract_exact_csv(content, name='download', depth=0):
    """Tam dosya adı eşleşir; full sürümü yanlışlıkla kabul etmez."""
    if depth > 3: raise ValueError('ZIP iç içe geçme sınırı aşıldı.')
    if name.lower().endswith('.csv') and Path(name).name.lower() == 'bank-additional.csv':
        return content
    if not zipfile.is_zipfile(io.BytesIO(content)):
        raise ValueError('Arşivde istenen bank-additional.csv bulunamadı.')
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        items = [p for p in z.namelist() if not p.endswith('/')]
        for p in items:
            if Path(p).name.lower() == 'bank-additional.csv':
                return z.read(p)
        for p in items:
            if p.lower().endswith('.zip'):
                try: return extract_exact_csv(z.read(p), p, depth + 1)
                except ValueError: pass
    raise ValueError('Arşivde bank-additional.csv yok (full.csv kabul edilmez).')

CSV_PATH = DATA_DIR / 'bank-additional.csv'
if not CSV_PATH.exists():
    errors = []
    for url in OFFICIAL_URLS:
        try:
            print('Veri indirme deneniyor:', url)
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=25) as r:
                downloaded = r.read()
            csv_bytes = extract_exact_csv(downloaded, url)
            CSV_PATH.write_bytes(csv_bytes)
            print('Resmî kaynaktan veri alındı.')
            break
        except Exception as e:
            errors.append(f'{url}: {e}')
    if not CSV_PATH.exists():
        try:
            from google.colab import files
            print('UCI bağlantısı başarısız. Lütfen bank-additional.csv veya arşivini yükleyin.')
            uploaded = files.upload()
            for filename, content in uploaded.items():
                try:
                    CSV_PATH.write_bytes(extract_exact_csv(content, filename))
                    print('Yüklendi:', filename)
                    break
                except ValueError: continue
        except ImportError:
            pass
    if not CSV_PATH.exists():
        raise RuntimeError('Veri indirilemedi. UCI sayfasından bank-additional.csv dosyasını indirip ' + str(CSV_PATH) + ' yoluna koyun. Denemeler: ' + str(errors))

raw = pd.read_csv(CSV_PATH, sep=';', encoding='utf-8')
print('Ham veri boyutu:', raw.shape)
if raw.shape != (4119, 21):
    raise ValueError(f'Beklenen ham boyut (4119, 21); bulunan: {raw.shape}. Yanlış dosya olabilir.')
if not {'y', 'duration', 'pdays'}.issubset(raw.columns):
    raise ValueError('Gerekli y, duration veya pdays sütunlarından biri eksik.')
if set(raw['y'].dropna().unique()) != {'yes', 'no'}:
    raise ValueError('Beklenen hedef sınıflar yes/no değil.')
print('Sütunlar:', ', '.join(raw.columns))
display(raw.head(5))
# Verinin gerçek baytlarının SHA-256 özeti kaydedilir; Git blob SHA ile karıştırılmaz.
raw_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()
digest_path = RESULTS / 'raw_data_sha256.json'
if digest_path.exists():
    older = json.loads(digest_path.read_text(encoding='utf-8'))
    if older.get('sha256') != raw_sha256:
        raise RuntimeError('Ham veri SHA-256 değişmiş: başka veri sürümünü karıştırmayın.')
else:
    digest_path.write_text(json.dumps({'file':'data/bank-additional.csv','sha256':raw_sha256,'rows':4119},indent=2),encoding='utf-8')
print('Ham dosya SHA-256:', raw_sha256)


### 3. Ham veri kalite kontrolü, kopya politikası ve stratified split
**Önceden sabitlenen kopya politikası:** Tamamen özdeş satırlar varsa, rastgele bölmeden **önce** yalnızca ilk kayıt tutulur ve düşürülen orijinal indeksler raporlanır. Böylece tam kopyanın hem development hem testte yer alması engellenir. Bu işlem tüm satırın birebir eşitliğine dayanır; hedefle ilişkili özellik seçimi yapılmaz.

`dataset_split.csv` dosyasında **orijinal indeks**, **development/test ataması** ve **bölümün sınıf oranları** kaydedilir; her test satırının gerçek hedef etiketi bu dosyada saklanmaz. Sınıf oranları bölme doğrulaması için mekanik olarak hesaplanır, model seçimi için kullanılmaz.

In [ ]:
# Veri setinin okunma sırası orijinal satır indeksidir.
raw.index.name = 'original_index'
raw_dups = raw.duplicated(keep='first')
removed_indices = raw.index[raw_dups].tolist()
print('Tam kopya satır sayısı:', len(removed_indices))
pd.DataFrame({'removed_original_index': removed_indices}).to_csv(RESULTS / 'removed_exact_duplicates.csv', index=False)
clean = raw.loc[~raw_dups].copy()

# duration kararı performans görülmeden önce alınır.
clean = clean.drop(columns=['duration'])
assert 'duration' not in clean.columns

indices = clean.index.to_numpy()
strata = clean['y'].map({'yes':1, 'no':0}).to_numpy()
DEV_IDX, TEST_IDX = train_test_split(indices, test_size=0.20, stratify=strata, random_state=SEEDS['split'])
assert not (set(DEV_IDX) & set(TEST_IDX))
DEV = clean.loc[DEV_IDX].copy()   # Bundan sonraki analizler YALNIZCA DEV kullanacak.

split_counts = {}
for split_name, chosen in [('development', DEV_IDX), ('test', TEST_IDX)]:
    rates = clean.loc[chosen, 'y'].value_counts(normalize=True)
    split_counts[split_name] = {
        'n': len(chosen),
        'yes_rate': float(rates.get('yes', 0.0)),
        'no_rate': float(rates.get('no', 0.0)),
    }

split_rows = []
for split_name, chosen in [('development', DEV_IDX), ('test', TEST_IDX)]:
    for idx in chosen:
        split_rows.append({
            'original_index': int(idx), 'split': split_name,
            'split_yes_rate': split_counts[split_name]['yes_rate'],
            'split_no_rate': split_counts[split_name]['no_rate']
        })
split_df = pd.DataFrame(split_rows).sort_values('original_index')
split_path = RESULTS / 'dataset_split.csv'
if split_path.exists():
    earlier = pd.read_csv(split_path).sort_values('original_index').reset_index(drop=True)
    proposed = split_df.reset_index(drop=True)
    pd.testing.assert_frame_equal(earlier, proposed, check_exact=False, rtol=0, atol=1e-12)
else:
    split_df.to_csv(split_path, index=False)
(RESULTS / 'split_summary.json').write_text(json.dumps({
    'raw_n': len(raw), 'removed_duplicate_n': len(removed_indices),
    'remaining_n': len(clean), 'seed': SEEDS['split'], 'splits': split_counts,
    'duplicate_policy': 'remove exact duplicate complete rows before split, retain first',
}, ensure_ascii=False, indent=2), encoding='utf-8')

print('Ham kayıt:', len(raw), '| Tekilleştirilmiş:', len(clean))
print('Development:', split_counts['development'])
print('Test:', split_counts['test'])
print('Test gözlemleri EDA için KULLANILMADI.')

### 4. EDA (yalnızca development) — veri türleri ve eksik/özel kodlu değerler
`unknown`, NaN'a dönüştürülmez. Ön işlemenin hangi çeşidinin daha iyi olacağına burada karar verilmeyecek; seçenekler daha sonra **inner CV** ile karşılaştırılacak.

In [ ]:
X_DEV = DEV.drop(columns=['y'])
y_dev = DEV['y'].map({'yes': 1, 'no': 0}).astype(int)
numeric_cols = X_DEV.select_dtypes(include='number').columns.tolist()
categorical_cols = X_DEV.select_dtypes(exclude='number').columns.tolist()
print('Girdi özellikleri:', X_DEV.shape[1], '| Sayısal:', len(numeric_cols), '| Kategorik:', len(categorical_cols))
print('Development satır sayısı:', len(DEV))

profile = pd.DataFrame({
    'column': X_DEV.columns,
    'dtype': [str(X_DEV[c].dtype) for c in X_DEV],
    'unique_values': [X_DEV[c].nunique(dropna=True) for c in X_DEV],
    'real_nan_count': [X_DEV[c].isna().sum() for c in X_DEV],
    'unknown_count': [(X_DEV[c].astype(str).str.strip().str.lower() == 'unknown').sum() for c in X_DEV],
})
profile.to_csv(RESULTS / 'eda_column_profile.csv', index=False)
display(profile)
print('Development tam kopya sayısı:', int(DEV.duplicated().sum()))
print('pdays=999 özel kod sayısı:', int((DEV['pdays'] == 999).sum()))

### 5. Sınıf dağılımı ve dengesizlik
Bankacılık örneğinde `no` sınıfı çoğunlukta olabilir. Bu yüzden yüksek Accuracy tek başına yeterli değildir; ödevin ana metriği **Average Precision (AP)** olarak belirlenmiştir.

In [ ]:
counts = DEV['y'].value_counts().reindex(['no', 'yes'], fill_value=0)
class_dist = pd.DataFrame({'class': counts.index, 'count': counts.values, 'percent': (counts.values / len(DEV) * 100).round(2)})
class_dist.to_csv(RESULTS / 'eda_class_distribution.csv', index=False)
display(class_dist)
fig, ax = plt.subplots(figsize=(7,4))
bars = ax.bar(class_dist['class'], class_dist['count'])
for bar, pct in zip(bars, class_dist['percent']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f'%{pct:.1f}', ha='center', va='bottom')
ax.set(title='Şekil 1. Development sınıf dağılımı', xlabel='Hedef sınıf', ylabel='Müşteri sayısı')
fig.tight_layout(); fig.savefig(FIGURES / '01_class_distribution.png', dpi=180); plt.show()

### 6. Sayısal değişkenler — ortalama, medyan, SD, IQR, min/maks
`pdays=999`, daha önce temas edilmediğini kodlar. Bu nedenle `pdays` dağılım istatistikleri, temas edilmiş müşteriler (`pdays != 999`) için ayrıca hesaplanır; `999` değerleri silinmez.

In [ ]:
summary_rows = []
for col in numeric_cols:
    s = pd.to_numeric(DEV[col], errors='coerce').dropna()
    is_pdays = (col == 'pdays')
    if is_pdays: s = s[s != 999]  # Yalnızca betimleyici analiz; veri satırı silinmez.
    if s.empty:
        summary_rows.append({'variable':col, 'n_analyzed':0, 'mean':np.nan, 'median':np.nan, 'sd':np.nan, 'IQR':np.nan, 'min':np.nan, 'max':np.nan, 'note':'pdays=999 excluded from descriptive numbers' if is_pdays else ''})
        continue
    q1,q3 = s.quantile([0.25,0.75])
    summary_rows.append({'variable':col, 'n_analyzed':len(s), 'mean':s.mean(), 'median':s.median(), 'sd':s.std(ddof=1), 'IQR':q3-q1, 'min':s.min(), 'max':s.max(), 'note':'pdays=999 excluded from descriptive numbers' if is_pdays else ''})
numeric_summary = pd.DataFrame(summary_rows)
numeric_summary.to_csv(RESULTS / 'eda_numeric_summary.csv', index=False)
display(numeric_summary.round(3))

In [ ]:
plot_cols = [c for c in ['age','campaign','euribor3m'] if c in numeric_cols]
if not plot_cols: plot_cols = [c for c in numeric_cols if c != 'pdays'][:3]
fig, axs = plt.subplots(len(plot_cols), 2, figsize=(12, 3.3 * len(plot_cols)), squeeze=False)
for i, col in enumerate(plot_cols):
    vals = DEV[col].dropna()
    axs[i,0].hist(vals, bins=25)
    axs[i,0].set(title=f'{col} — histogram', xlabel=col, ylabel='Frekans')
    axs[i,1].boxplot(vals, vert=False)
    axs[i,1].set(title=f'{col} — boxplot', xlabel=col)
fig.suptitle('Şekil 2. Seçilmiş sayısal dağılımlar (development)', fontsize=14)
fig.tight_layout(); fig.savefig(FIGURES / '02_numeric_distributions.png', dpi=180); plt.show()

### 7. Kategorik değişkenler — frekans ve `unknown` sayımı

In [ ]:
cat_freq_list = []
for c in categorical_cols:
    value_counts = DEV[c].value_counts(dropna=False)
    for v, n in value_counts.items():
        cat_freq_list.append({'variable':c, 'category':str(v), 'count':int(n), 'percent':100 * n/len(DEV)})
cat_freq = pd.DataFrame(cat_freq_list)
cat_freq.to_csv(RESULTS / 'eda_categorical_frequencies.csv', index=False)
display(cat_freq.groupby('variable', group_keys=False).head(5))

plot_cats = [c for c in ['job', 'education', 'contact'] if c in categorical_cols]
if not plot_cats: plot_cats = categorical_cols[:3]
fig, axs = plt.subplots(len(plot_cats), 1, figsize=(11,3.5*len(plot_cats)), squeeze=False)
for i, c in enumerate(plot_cats):
    counts_c = DEV[c].value_counts().head(12)
    axs[i,0].barh(counts_c.index.astype(str)[::-1], counts_c.values[::-1])
    axs[i,0].set(title=c, xlabel='Müşteri sayısı')
fig.suptitle('Şekil 3. Kategorik değişkenler (development)', fontsize=14)
fig.tight_layout(); fig.savefig(FIGURES / '03_categorical_distributions.png', dpi=180); plt.show()

### 8. Korelasyon matrisi ve hedefe göre betimsel karşılaştırma
Bu inceleme **yalnızca hipotez üretir**. Burada gördüğümüz ilişkilere dayanarak özellik elemek istersek, kararı sonraki notebook'ta **her outer-train içindeki inner CV ile yeniden** almamız gerekir.

In [ ]:
corr = DEV[numeric_cols].corr(numeric_only=True)
corr.to_csv(RESULTS / 'eda_numeric_correlations.csv')
fig, ax = plt.subplots(figsize=(11,8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f', ax=ax, square=False)
ax.set(title='Şekil 4. Sayısal özellik korelasyonları (development)')
fig.tight_layout(); fig.savefig(FIGURES / '04_correlation_matrix.png', dpi=180); plt.show()

selected = [c for c in ['age', 'campaign', 'euribor3m'] if c in numeric_cols]
by_target = DEV.groupby('y')[selected].agg(['count','mean','median','std'])
by_target.to_csv(RESULTS / 'eda_target_group_comparison.csv')
display(by_target.round(3))

### 9. Aykırı değerlerin işaretlenmesi (satır silmeyin)
IQR sınırları araştırma amaçlı hesaplanır. `pdays=999` bu analizde dışarıda bırakılır çünkü **özel kod**, yanlış ölçüm değil. İlgili veri üzerinde kırpma (clipping) uygulanıp uygulanmayacağına sonraki aşamada inner CV karar verecek.

In [ ]:
outlier_rows = []
for c in numeric_cols:
    s = pd.to_numeric(DEV[c], errors='coerce')
    analyzed = s[s != 999] if c == 'pdays' else s
    analyzed = analyzed.dropna()
    if analyzed.empty: continue
    q1, q3 = analyzed.quantile([0.25,0.75]); iqr=q3-q1
    lo, hi = q1-1.5*iqr, q3+1.5*iqr
    count = int(((analyzed < lo) | (analyzed > hi)).sum())
    outlier_rows.append({'variable':c, 'q1':q1, 'q3':q3, 'iqr':iqr, 'lower_fence':lo, 'upper_fence':hi, 'flagged_count':count, 'analyzed_count':len(analyzed), 'note':'999 excluded as special code' if c=='pdays' else ''})
outlier_table=pd.DataFrame(outlier_rows)
outlier_table.to_csv(RESULTS / 'eda_outlier_flags.csv', index=False)
display(outlier_table.round(2))

### 10. En az üç test edilebilir hipotezi **önceden** kaydetme
Aşağıdaki hipotezler, test setini görmeden tanımlanır. Hangi hipotezin desteklendiği daha sonra **gerçek nested CV sonuçlarına** göre tartışılacaktır. Bu aşamada birinin doğru olduğunu iddia etmiyoruz.

In [ ]:
hypotheses = [
    {'id':'H1', 'hypothesis':'StandardScaler ve MinMaxScaler, aynı MLP ve eğitim ayarlarında farklı AP başarısı verebilir.', 'comparison':'M1 vs M4'},
    {'id':'H2', 'hypothesis':'İki gizli katman (64,32), tek gizli katmana (32,) göre AP ve yakınsama davranışını değiştirebilir.', 'comparison':'M1 vs M5'},
    {'id':'H3', 'hypothesis':'Learning-rate scheduler kullanılması, sabit öğrenme hızına göre AP ve en iyi epoch üzerinde farklılık oluşturabilir.', 'comparison':'M1 vs M6'},
    {'id':'H4', 'hypothesis':'Aynı yapıdaki SGD, Adam ve AdamW optimizasyon algoritmalarının AP ve eğitim süresi farklı olabilir.', 'comparison':'M1 vs M2 vs M3'},
]
(CONFIGS / 'eda_hypotheses.json').write_text(json.dumps(hypotheses, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame(hypotheses))

### 11. Aşama 1 çıktılarını doğrulama ve kontrol özeti

Kod bütün çıktıları Google Drive altında saklar. ZIP ayrıca Drive'da oluşturulur; otomatik indirme yapılmaz. Sonraki aşamada aynı `dataset_split.csv` kullanılmalıdır.

In [ ]:
expected = [
    RESULTS/'dataset_split.csv', RESULTS/'split_summary.json', RESULTS/'raw_data_sha256.json',
    RESULTS/'eda_column_profile.csv', RESULTS/'eda_class_distribution.csv',
    RESULTS/'eda_numeric_summary.csv', RESULTS/'eda_categorical_frequencies.csv',
    RESULTS/'eda_numeric_correlations.csv', RESULTS/'eda_outlier_flags.csv',
    CONFIGS/'seeds.json', CONFIGS/'eda_hypotheses.json',
    FIGURES/'01_class_distribution.png', FIGURES/'02_numeric_distributions.png',
    FIGURES/'03_categorical_distributions.png', FIGURES/'04_correlation_matrix.png',
]
missing = [str(p) for p in expected if not p.exists()]
if missing: raise RuntimeError('Eksik çıktılar: ' + str(missing))

# Çıktıların denetlenebilir hash manifesti; test etiketlerini dışarı vermez.
files_to_hash = sorted([p for top in [CONFIGS, RESULTS, FIGURES] for p in top.rglob('*') if p.is_file() and p.name != 'stage1_sha256_manifest.json'])
manifest = {'base_seed': BASE_SEED, 'hash_type': 'SHA-256 (raw file bytes)',
            'files': {str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in files_to_hash}}
(RESULTS/'stage1_sha256_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
zip_path = ROOT/'YSA_Asama1_Sonuclar_31040.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for base in [CONFIGS, RESULTS, FIGURES]:
        for p in base.rglob('*'):
            if p.is_file(): archive.write(p, p.relative_to(ROOT))
print('AŞAMA 1 BAŞARILI —', len(expected), 'zorunlu çıktı kontrol edildi.')
print('Drive klasörü:', ROOT)
print('ZIP:', zip_path)
print('SHA manifest:', RESULTS/'stage1_sha256_manifest.json')
print('AŞAMA 2 için: split ve seed dosyalarını DEĞİŞTİRMEYİN.')
